# Agent with Planning Loop
A ReAct-style agent: Plan → Execute → Observe → Reflect, looping until done.
Uses your existing OpenRouter setup.

In [ ]:
import os, json, textwrap
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url='https://openrouter.ai/api/v1',
)

MODEL = 'openrouter/owl-alpha'  # swap to gpt-4o, claude-3-5-sonnet, etc.
MAX_ITERATIONS = 6              # safety ceiling on the loop

In [ ]:
# ── Tools the agent can call ──────────────────────────────────────────────────
# Add real implementations here (web search, code exec, DB query, etc.)

def tool_search(query: str) -> str:
    """Simulated search tool. Replace with a real API call."""
    return f"[Search result for '{query}']: Found relevant information about {query}."

def tool_calculate(expression: str) -> str:
    """Safe arithmetic evaluator."""
    try:
        # Only allow safe math expressions
        allowed = set('0123456789+-*/()., ')
        if not all(c in allowed for c in expression):
            return "Error: unsafe expression"
        result = eval(expression, {'__builtins__': {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def tool_summarise(text: str) -> str:
    """Summarise text using the LLM."""
    resp = client.chat.completions.create(
        model=MODEL,
        max_tokens=200,
        messages=[{"role": "user", "content": f"Summarise this in 2 sentences:\n{text}"}]
    )
    return resp.choices[0].message.content

# Registry: name -> (callable, description)
TOOLS = {
    "search":    (tool_search,    "Search the web for information. Input: query string."),
    "calculate": (tool_calculate, "Evaluate a math expression. Input: expression string."),
    "summarise": (tool_summarise, "Summarise a block of text. Input: text string."),
}

def run_tool(name: str, input_str: str) -> str:
    if name not in TOOLS:
        return f"Unknown tool '{name}'"
    fn, _ = TOOLS[name]
    return fn(input_str)

In [ ]:
# ── System prompt ─────────────────────────────────────────────────────────────

def build_system_prompt() -> str:
    tool_descriptions = '\n'.join(
        f"  - {name}: {desc}" for name, (_, desc) in TOOLS.items()
    )
    return textwrap.dedent(f"""
        You are a goal-oriented AI agent. You work in a loop:
        PLAN → EXECUTE → OBSERVE → REFLECT → repeat until done.

        Available tools:
        {tool_descriptions}

        At each step respond with EXACTLY one JSON object (no markdown, no extra text):

        If you need to call a tool:
        {{"action": "tool", "tool": "<name>", "input": "<input>", "thought": "<why>"}}

        If you have the final answer:
        {{"action": "finish", "answer": "<final answer>", "thought": "<summary>"}}

        Rules:
        - Use tools when you need external information or computation.
        - Finish as soon as you have enough information to answer fully.
        - Never guess when a tool can give you the real answer.
    """).strip()

print(build_system_prompt())

In [ ]:
# ── Core agent loop ───────────────────────────────────────────────────────────

def run_agent(goal: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system",  "content": build_system_prompt()},
        {"role": "user",    "content": f"Goal: {goal}"},
    ]

    print(f"\n{'='*60}")
    print(f"GOAL: {goal}")
    print('='*60)

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n── Iteration {iteration} ──")

        # LLM decides next action
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=500,
            messages=messages,
        )
        raw = response.choices[0].message.content.strip()

        # Parse JSON response
        try:
            step = json.loads(raw)
        except json.JSONDecodeError:
            # If the model wrapped it in markdown fences, strip them
            cleaned = raw.strip('`').replace('json\n', '', 1)
            step = json.loads(cleaned)

        thought = step.get('thought', '')
        print(f"  Thought : {thought}")

        if step['action'] == 'finish':
            print(f"\n✅ DONE — {step['answer']}")
            return step['answer']

        # Tool call
        tool_name  = step['tool']
        tool_input = step['input']
        print(f"  Tool    : {tool_name}({tool_input!r})")

        observation = run_tool(tool_name, tool_input)
        print(f"  Result  : {observation}")

        # Append the exchange to history
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user",      "content": f"Observation: {observation}"})

    return "Max iterations reached without a final answer."

In [ ]:
# ── Run it! ───────────────────────────────────────────────────────────────────

answer = run_agent(
    "What is the GDP of Nigeria, and what is 15% of that number?"
)

print(f"\nFinal answer: {answer}")

In [ ]:
# ── Try another goal ──────────────────────────────────────────────────────────

answer2 = run_agent(
    "Find information about quantum computing and give me a 2-sentence summary."
)
print(f"\nFinal answer: {answer2}")